In [0]:
%run ./common/_utils_config

In [0]:
%run ./common/_utils_bronze_framework

In [0]:
%run ./common/_utils_onboarding_tools

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
print(f"Schema ready: {catalog}.{schema}")

In [0]:
# a) Broken YAML: unquoted {start}
bad_yaml = """
source_name: test_bad_yaml
source_type: http_api
description: test
incremental:
  type: date_range
  days_back: 7
  chunk_days: 7
endpoints:
  - name: prices
    url: https://apidatos.ree.es/es/datos/mercados/precios-mercados-tiempo-real
    response_format: json
    params:
      start_date: {start}T00:00
      end_date: {end}T23:59
"""
print("a)", json.loads(propose_source_spec(bad_yaml))["stage"], "\n")

# b) Valid YAML, but the source already exists
print("b)", json.loads(propose_source_spec(spec_to_yaml(load_source_spec_file("ree"))))["errors"], "\n")

# c) Valid spec, but the API path does not exist -> fails in the dry run
not_found_yaml = """
source_name: test_not_found
source_type: http_api
description: "Test: endpoint that does not exist"
incremental:
  type: snapshot
endpoints:
  - name: nothing
    url: https://date.nager.at/api/v3/ThisDoesNotExist
    method: GET
    response_format: json
"""
print("c)", json.loads(propose_source_spec(not_found_yaml))["stage"])

In [0]:
holidays_yaml = """
source_name: public_holidays_es
source_type: http_api
description: "Nager.Date public holidays API: national and regional public holidays in Spain for 2026."

auth:
  type: none

incremental:
  type: snapshot

endpoints:
  - name: holidays_2026
    url: https://date.nager.at/api/v3/PublicHolidays/2026/ES
    method: GET
    response_format: json
"""

print(propose_source_spec(holidays_yaml))

In [0]:
print("Source specs:", list_source_specs())

display(
    spark.table(bronze_table_name("public_holidays_es", "holidays_2026"))
    .select("source_file", "chunk_start", "run_id",
            F.expr("payload:[0].date::string").alias("first_date"),
            F.expr("payload:[0].localName::string").alias("first_holiday"),
            F.expr("array_size(cast(payload as array<variant>))").alias("holidays_in_file"))
)